<a href="https://colab.research.google.com/github/Shamit-butterchicken/Adaptive-Dashboard-HCI/blob/main/lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🥙 LSTM on Recipe Data

In this notebook, we'll walk through the steps required to train your own LSTM on the recipes dataset

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import json
import re
import string

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, losses

## 0. Parameters <a name="parameters"></a>

In [ ]:
VOCAB_SIZE = 10000
MAX_LEN = 200
EMBEDDING_DIM = 100
N_UNITS = 128
VALIDATION_SPLIT = 0.2
SEED = 42
LOAD_MODEL = False
BATCH_SIZE = 32
EPOCHS = 25

## 1. Load the data <a name="load"></a>

In [ ]:
# Load the full dataset
with open("/app/data/epirecipes/full_format_recipes.json") as json_data:
    recipe_data = json.load(json_data)

In [ ]:
# Filter the dataset
filtered_data = [
    "Recipe for " + x["title"] + " | " + " ".join(x["directions"])
    for x in recipe_data
    if "title" in x
    and x["title"] is not None
    and "directions" in x
    and x["directions"] is not None
]

In [ ]:
# Count the recipes
n_recipes = len(filtered_data)
print(f"{n_recipes} recipes loaded")

In [ ]:
example = filtered_data[9]
print(example)

## 2. Tokenise the data

In [ ]:
# Pad the punctuation, to treat them as separate 'words'
def pad_punctuation(s):
    s = re.sub(f"([{string.punctuation}])", r" \1 ", s)
    s = re.sub(" +", " ", s)
    return s


text_data = [pad_punctuation(x) for x in filtered_data]

In [ ]:
# Display an example of a recipe
example_data = text_data[9]
example_data

In [ ]:
# Convert to a Tensorflow Dataset
text_ds = (
    tf.data.Dataset.from_tensor_slices(text_data)
    .batch(BATCH_SIZE)
    .shuffle(1000)
)

In [ ]:
# Create a vectorisation layer
vectorize_layer = layers.TextVectorization(
    standardize="lower",
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=MAX_LEN + 1,
)

In [ ]:
# Adapt the layer to the training set
vectorize_layer.adapt(text_ds)
vocab = vectorize_layer.get_vocabulary()

In [ ]:
# Display some token:word mappings
for i, word in enumerate(vocab[:10]):
    print(f"{i}: {word}")

In [ ]:
# Display the same example converted to ints
example_tokenised = vectorize_layer(example_data)
print(example_tokenised.numpy())

## 3. Create the Training Set

In [ ]:
# Create the training set of recipes and the same text shifted by one word
def prepare_inputs(text):
    text = tf.expand_dims(text, -1)
    tokenized_sentences = vectorize_layer(text)
    x = tokenized_sentences[:, :-1]
    y = tokenized_sentences[:, 1:]
    return x, y


train_ds = text_ds.map(prepare_inputs)

## 4. Build the LSTM <a name="build"></a>

In [ ]:
inputs = layers.Input(shape=(None,), dtype="int32")
x = layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM)(inputs)
x = layers.LSTM(N_UNITS, return_sequences=True)(x)
outputs = layers.Dense(VOCAB_SIZE, activation="softmax")(x)
lstm = models.Model(inputs, outputs)
lstm.summary()

In [ ]:
if LOAD_MODEL:
    # model.load_weights('./models/model')
    lstm = models.load_model("./models/lstm", compile=False)

## 5. Train the LSTM <a name="train"></a>

In [ ]:
loss_fn = losses.SparseCategoricalCrossentropy()
lstm.compile("adam", loss_fn)

In [ ]:
# Create a TextGenerator checkpoint
class TextGenerator(callbacks.Callback):
    def __init__(self, index_to_word, top_k=10):
        self.index_to_word = index_to_word
        self.word_to_index = {
            word: index for index, word in enumerate(index_to_word)
        }  # <1>

    def sample_from(self, probs, temperature):  # <2>
        probs = probs ** (1 / temperature)
        probs = probs / np.sum(probs)
        return np.random.choice(len(probs), p=probs), probs

    def generate(self, start_prompt, max_tokens, temperature):
        start_tokens = [
            self.word_to_index.get(x, 1) for x in start_prompt.split()
        ]  # <3>
        sample_token = None
        info = []
        while len(start_tokens) < max_tokens and sample_token != 0:  # <4>
            x = np.array([start_tokens])
            y = self.model.predict(x, verbose=0)  # <5>
            sample_token, probs = self.sample_from(y[0][-1], temperature)  # <6>
            info.append({"prompt": start_prompt, "word_probs": probs})
            start_tokens.append(sample_token)  # <7>
            start_prompt = start_prompt + " " + self.index_to_word[sample_token]
        print(f"\ngenerated text:\n{start_prompt}\n")
        return info

    def on_epoch_end(self, epoch, logs=None):
        self.generate("recipe for", max_tokens=100, temperature=1.0)

In [ ]:
# Create a model save checkpoint
model_checkpoint_callback = callbacks.ModelCheckpoint(
    filepath="./checkpoint/checkpoint.ckpt",
    save_weights_only=True,
    save_freq="epoch",
    verbose=0,
)

tensorboard_callback = callbacks.TensorBoard(log_dir="./logs")

# Tokenize starting prompt
text_generator = TextGenerator(vocab)

In [ ]:
lstm.fit(
    train_ds,
    epochs=EPOCHS,
    callbacks=[model_checkpoint_callback, tensorboard_callback, text_generator],
)

In [ ]:
# Save the final model
lstm.save("./models/lstm")

## 6. Generate text using the LSTM

In [ ]:
def print_probs(info, vocab, top_k=5):
    for i in info:
        print(f"\nPROMPT: {i['prompt']}")
        word_probs = i["word_probs"]
        p_sorted = np.sort(word_probs)[::-1][:top_k]
        i_sorted = np.argsort(word_probs)[::-1][:top_k]
        for p, i in zip(p_sorted, i_sorted):
            print(f"{vocab[i]}:   \t{np.round(100*p,2)}%")
        print("--------\n")

In [ ]:
info = text_generator.generate(
    "recipe for roasted vegetables | chop 1 /", max_tokens=10, temperature=1.0
)

In [ ]:
print_probs(info, vocab)

In [ ]:
info = text_generator.generate(
    "recipe for roasted vegetables | chop 1 /", max_tokens=10, temperature=0.2
)

In [ ]:
print_probs(info, vocab)

In [ ]:
info = text_generator.generate(
    "recipe for chocolate ice cream |", max_tokens=7, temperature=1.0
)
print_probs(info, vocab)

In [ ]:
info = text_generator.generate(
    "recipe for chocolate ice cream |", max_tokens=7, temperature=0.2
)
print_probs(info, vocab)

In [2]:
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Load raw JSON dataset
with open("full_format_recipes.json", "r") as f:
    raw_recipes = json.load(f)

# 2. Clean incomplete entries
def is_valid_recipe(r):
    return isinstance(r, dict) and r.get("title") and r.get("ingredients") and r.get("directions")

valid_recipes = [r for r in raw_recipes if is_valid_recipe(r)]

def format_recipe(r):
    title = r["title"].strip()
    ingredients = " ".join([i.strip() for i in r["ingredients"] if i])
    directions = " ".join([d.strip() for d in r["directions"] if d])
    return f"recipe start title : {title} ingredients : {ingredients} directions : {directions} recipe end"

recipe_texts = [format_recipe(r) for r in valid_recipes]

# 3. Setup Vectorization layer
VOCAB_SIZE = 10000
SEQ_LEN = 40
BATCH_SIZE = 64

vectorize_layer = layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=SEQ_LEN + 1,
    standardize="lower_and_strip_punctuation"
)

vectorize_layer.adapt(recipe_texts)

# 4. Prepare TensorFlow input/target dataset pairs
text_ds = tf.data.Dataset.from_tensor_slices(recipe_texts)
vectorized_ds = text_ds.map(lambda x: vectorize_layer(x))

def split_input_target(seq):
    return seq[:-1], seq[1:]

dataset = vectorized_ds.map(split_input_target)
dataset = dataset.shuffle(10000).batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

print(f"Data ready. Total cleaned recipes: {len(recipe_texts)}")

Data ready. Total cleaned recipes: 20043


In [3]:
EMBEDDING_DIM = 256
LSTM_UNITS = 512

# 1. Define network architecture matching textbook setup
inputs = layers.Input(shape=(SEQ_LEN,), dtype=tf.int64)
x = layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM)(inputs)
x = layers.LSTM(LSTM_UNITS, return_sequences=True)(x)
outputs = layers.Dense(VOCAB_SIZE)(x)

model = models.Model(inputs, outputs)

# 2. Compile model with cross-entropy loss
model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer="adam"
)

# 3. Train the model for 10 epochs
print("Starting model training...")
history = model.fit(dataset, epochs=10)

Starting model training...
Epoch 1/10
 71/313 ━━━━━━━━━━━━━━━━━━━━ 26:30 7s/step - loss: 7.2505

KeyboardInterrupt: 

In [3]:
EMBEDDING_DIM = 256
LSTM_UNITS = 256  # Reduced from 512 for faster training

inputs = layers.Input(shape=(SEQ_LEN,), dtype=tf.int64)
x = layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM)(inputs)
x = layers.LSTM(LSTM_UNITS, return_sequences=True)(x)
outputs = layers.Dense(VOCAB_SIZE)(x)

model = models.Model(inputs, outputs)

model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer="adam"
)

# Reduced to 3 epochs for speed
print("Starting fast training...")
history = model.fit(dataset, epochs=3)

Starting fast training...
Epoch 1/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 18s 34ms/step - loss: 5.5068
Epoch 2/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 13s 35ms/step - loss: 4.2609
Epoch 3/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 13s 35ms/step - loss: 3.6735


In [4]:
import numpy as np

def sample_with_temperature(logits, temperature):
    """Rescales logits by temperature and samples from the categorical distribution."""
    logits = np.asarray(logits).astype("float64") / temperature
    exp_logits = np.exp(logits - np.max(logits))  # Subtract max for numerical stability
    probs = exp_logits / np.sum(exp_logits)
    return np.random.choice(len(probs), p=probs)

def generate_recipe(model, vectorize_layer, prompt, num_words_to_generate, temperature):
    vocab = vectorize_layer.get_vocabulary()

    # Convert seed text prompt to token IDs
    prompt_tokens = vectorize_layer([prompt]).numpy()[0]
    prompt_tokens = [t for t in prompt_tokens if t != 0] # Remove padding zeros

    generated_tokens = list(prompt_tokens)

    for _ in range(num_words_to_generate):
        # Maintain sequence window of SEQ_LEN
        if len(generated_tokens) < SEQ_LEN:
            input_seq = [0] * (SEQ_LEN - len(generated_tokens)) + generated_tokens
        else:
            input_seq = generated_tokens[-SEQ_LEN:]

        input_tensor = tf.expand_dims(input_seq, 0)
        predictions = model(input_tensor, training=False)

        # Get logits for the predicted next word (last sequence position)
        last_token_logits = predictions[0, -1, :].numpy()

        # Sample token ID using scaled temperature
        next_token = sample_with_temperature(last_token_logits, temperature)
        generated_tokens.append(next_token)

    # Reconstruct generated token IDs back into string words
    generated_words = [vocab[idx] if idx < len(vocab) else "[UNK]" for idx in generated_tokens]
    return " ".join(generated_words)

print("Generator function ready!")

Generator function ready!


In [5]:
prompt_text = "recipe start title : chocolate cake"
words_to_generate = 70

# Output 1: Temperature = 0.4
print("=== GENERATED OUTPUT AT TEMPERATURE = 0.4 ===")
output_04 = generate_recipe(model, vectorize_layer, prompt_text, words_to_generate, temperature=0.4)
print(output_04)

print("\n" + "="*60 + "\n")

# Output 2: Temperature = 0.75
print("=== GENERATED OUTPUT AT TEMPERATURE = 0.75 ===")
output_075 = generate_recipe(model, vectorize_layer, prompt_text, words_to_generate, temperature=0.75)
print(output_075)

=== GENERATED OUTPUT AT TEMPERATURE = 0.4 ===
recipe start title chocolate cake with [UNK] ingredients 2 tablespoons extravirgin olive oil 1 teaspoon salt 1 teaspoon ground pepper 2 tablespoons extravirgin olive oil 2 tablespoons fresh lemon juice 1 cup water 1 tablespoon minced fresh ginger 2 tablespoons fresh lemon juice 2 tablespoons fresh lime juice 1 teaspoon salt 12 teaspoon ground cumin 1 tablespoon fresh lemon juice 1 teaspoon ground cinnamon 12 teaspoon ground black pepper 12 cup dry white wine vinegar


=== GENERATED OUTPUT AT TEMPERATURE = 0.75 ===
recipe start title chocolate cake with two [UNK] ingredients 1 teaspoon dried cumin 12 cup heavy cream 1 tablespoon fresh lemon juice 3 tablespoons olive oil 1 teaspoon salt 2 12 teaspoons rinsed peanuts 1 teaspoon salt 6 tablespoons plus 13 cup chopped fresh rosemary leaves 2 teaspoons minced fresh rosemary 1 12 teaspoons ground black pepper 2 cups 6 large eggs 1 12 tablespoons fresh lemon juice 1 teaspoon salt freshly ground bla